In [ ]:
from pathlib import Path

import numpy as np
import torch

from src.io import PipelinePaths, open_sample
from trackastra.model import Trackastra

In [ ]:
paths = PipelinePaths.discover()

# Change this only if you want to test another processed sample.
SAMPLE_ID = "44b6_0113de3b"

stage6_root = paths.processed_dataset(SAMPLE_ID)
segmentation_dir = stage6_root / "segmentation"

print("Sample:", SAMPLE_ID)
print("Stage 6:", stage6_root)
print("Raw Zarr:", paths.sample_zarr(SAMPLE_ID))

In [ ]:
segmentation_files = sorted(segmentation_dir.glob("t*.npy"))

if not segmentation_files:
    raise FileNotFoundError(
        f"No Stage 6 segmentations found in {segmentation_dir}"
    )

print("Stage 6 frames:", len(segmentation_files))

masks = np.stack(
    [
        np.load(path, allow_pickle=False)
        for path in segmentation_files
    ],
    axis=0,
)

print("Masks:", masks.shape, masks.dtype)

In [ ]:
raw = open_sample(paths.sample_zarr(SAMPLE_ID))

T = len(segmentation_files)

imgs = np.asarray(raw[:T])

print("Images:", imgs.shape, imgs.dtype)

In [ ]:
assert T == 20, f"Expected 20 frames, found {T}"
assert imgs.shape == masks.shape
assert imgs.ndim == 4
assert masks.ndim == 4

print()
print("Trackastra input ready:")
print("imgs :", imgs.shape, imgs.dtype)
print("masks:", masks.shape, masks.dtype)

In [ ]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model = Trackastra.from_pretrained(
    "ctc",
    device="cuda",
)

track_graph, masks_tracked = model.track(
    imgs=imgs,
    masks=masks,
    mode="greedy",
    n_workers=0,
    batch_size=4,
)

In [ ]:
print()
print("## Trackastra result")
print()

print("Tracked masks:", masks_tracked.shape, masks_tracked.dtype)
print("Graph nodes:", track_graph.number_of_nodes())
print("Graph edges:", track_graph.number_of_edges())

print()
print("## GPU memory")
print()

print(
    f"Peak allocated: "
    f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)

In [ ]:
original_counts = [
    np.count_nonzero(np.unique(frame))
    for frame in masks
]

tracked_counts = [
    np.count_nonzero(np.unique(frame))
    for frame in masks_tracked
]

print("Original instance counts:")
print(original_counts)

print()
print("Tracked-mask instance counts:")
print(tracked_counts)

## Exact Stage 9 visualization adapter

From this point onward, the notebook does **not** implement a separate
Trackastra visualization algorithm.

Instead, Trackastra's graph is converted to the ordinary Stage 9 track-table
schema:

`track_id, frame, z, y, x`

and the repository's current Stage 9 preparation and Napari layer code are
reused directly.

This deliberately keeps the comparison under the same conditions as the
existing tracker:

- the same nearest-Stage-6-cell assignment;
- the same 4.0 µm boundary filtering;
- the same new/ended track definitions;
- the same special-case track + centroid layers;
- the same hidden `Tracks - all` and `Centroids - all` layers.

No Trackastra-specific division or temporal-gap layers are injected into the
Stage 9 failure view.

In [ ]:

import numpy as np
import pandas as pd
from importlib import import_module, reload

from src.api import prepare_visualization_data
from src.io import load_npy_time_series, load_processed_dataset_inputs
from trackastra.tracking import graph_to_napari_tracks


# ------------------------------------------------------------
# 1. Convert Trackastra graph to the ordinary Stage 9 track schema
# ------------------------------------------------------------
#
# IMPORTANT:
# Use graph_to_napari_tracks directly rather than graph_to_ctc().
#
# graph_to_ctc() deliberately creates new CTC tracklets around temporal gaps,
# which would make a valid Trackastra gap-closing edge look like one ended
# track plus one new track to Stage 9.
#
# graph_to_napari_tracks() preserves those graph edges in the trajectory
# representation and is therefore the closer Stage 9 comparison.
# ------------------------------------------------------------

trackastra_tracks_array, trackastra_lineage_graph, _ = (
    graph_to_napari_tracks(track_graph)
)

tracks = pd.DataFrame(
    np.asarray(trackastra_tracks_array, dtype=float),
    columns=["track_id", "frame", "z", "y", "x"],
)

tracks["track_id"] = tracks["track_id"].astype(int)
tracks["frame"] = tracks["frame"].astype(int)


# ------------------------------------------------------------
# 2. Load the SAME Stage 6 cell tables used by Stage 9
# ------------------------------------------------------------

inputs = load_processed_dataset_inputs(
    SAMPLE_ID,
    paths=paths,
)

cells = pd.concat(
    [
        frame.assign(frame=index)
        for index, frame in enumerate(inputs.time_frames)
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 3. Reuse the CURRENT Stage 9 preparation code unchanged
# ------------------------------------------------------------

BOUNDARY_MARGIN_UM = 4.0
SHOW_BOUNDARY_TRACKS = False

endpoint_helpers = reload(
    import_module("src.09_visualization.step02_endpoints")
)

VOXEL_SIZE_ZYX = endpoint_helpers.VOXEL_SIZE_ZYX

visualization = prepare_visualization_data(
    tracks,
    cells,
)

endpoint_groups = endpoint_helpers.prepare_endpoint_track_groups(
    visualization.tracks,
    cells,
    imgs.shape[-3:],
    voxel_size_zyx=VOXEL_SIZE_ZYX,
    boundary_margin_um=BOUNDARY_MARGIN_UM,
)

SCALE_TZYX = (
    1.0,
    *VOXEL_SIZE_ZYX,
)


# ------------------------------------------------------------
# 4. Load the same Stage 6 visualization volumes
# ------------------------------------------------------------

preprocessed, _ = load_npy_time_series(
    inputs.root / "preprocessing",
    expected_frames=T,
)

binary_mask, _ = load_npy_time_series(
    inputs.root / "masking",
    expected_frames=T,
)

instance_labels, _ = load_npy_time_series(
    inputs.root / "segmentation",
    expected_frames=T,
)

# The raw 20-frame array already loaded for Trackastra.
raw = imgs


print("Voxel size ZYX:", VOXEL_SIZE_ZYX)
print("Trackastra track rows:", len(visualization.tracks))
print("Trackastra track IDs:", visualization.tracks["track_id"].nunique())
print(
    "New failure candidates:",
    endpoint_groups.new_failure_tracks["track_id"].nunique(),
)
print(
    "Ended failure candidates:",
    endpoint_groups.ended_failure_tracks["track_id"].nunique(),
)
print(
    "Boundary entries:",
    endpoint_groups.boundary_entry_tracks["track_id"].nunique(),
)
print(
    "Boundary exits:",
    endpoint_groups.boundary_exit_tracks["track_id"].nunique(),
)


## Stage 9 Napari viewer

The following cell intentionally mirrors the current `09_visualization.ipynb`
layer construction. Special-case trajectories and their matching centroid
points are added through the repository's own `add_track_group(...)` helper.

In [ ]:

import napari


# Reuse the exact current Stage 9 track-group layer helper.
napari_layers = reload(
    import_module("src.09_visualization.napari_layers")
)
add_track_group = napari_layers.add_track_group


viewer = napari.Viewer(ndisplay=3)

raw_contrast_limits = [
    float(np.percentile(raw, 1)),
    float(np.percentile(raw, 99.8)),
]


# ------------------------------------------------------------
# Same image/label layers as Stage 9
# ------------------------------------------------------------

viewer.add_image(
    raw,
    name="Raw Volume",
    scale=SCALE_TZYX,
    rendering="mip",
    colormap="gray",
    contrast_limits=raw_contrast_limits,
)

viewer.add_image(
    preprocessed,
    name="Preprocessed Volume",
    scale=SCALE_TZYX,
    rendering="mip",
    colormap="gray",
    contrast_limits=(0.0, 1.0),
    visible=False,
)

viewer.add_labels(
    binary_mask,
    name="Binary Mask",
    scale=SCALE_TZYX,
    visible=False,
)

viewer.add_labels(
    instance_labels,
    name="Instance Labels",
    scale=SCALE_TZYX,
    visible=False,
)


# ------------------------------------------------------------
# Same hidden "all tracks" layer as Stage 9
# ------------------------------------------------------------

all_tracks_layer = viewer.add_tracks(
    visualization.tracks_array,
    name="Tracks - all",
    scale=SCALE_TZYX,
    tail_length=20,
)
all_tracks_layer.visible = False


# ------------------------------------------------------------
# Same hidden "all centers" layer as Stage 9
#
# Cell ID is the visible text.
# Track ID remains available as a point property.
# ------------------------------------------------------------

all_centers_layer = viewer.add_points(
    visualization.points_array,
    name="Centroids - all",
    scale=SCALE_TZYX,
    size=4,
    face_color="red",
    properties={
        "track_id": visualization.track_ids,
        "cell_id": visualization.tracks["cell_id"].to_numpy(),
    },
    text={
        "string": "{cell_id}",
        "size": 8,
        "color": "white",
        "anchor": "center",
    },
)
all_centers_layer.visible = False


# ------------------------------------------------------------
# EXACT Stage 9 special-case groups
#
# add_track_group() creates BOTH:
#   1. the full selected trajectories
#   2. every matching centroid observation
#
# Therefore a special-case trajectory and its centers come from
# exactly the same DataFrame rows.
# ------------------------------------------------------------

add_track_group(
    viewer,
    endpoint_groups.ended_failure_tracks,
    track_name="Ended Tracks",
    point_name="Ended Centroids",
    color="red",
    scale=SCALE_TZYX,
)

add_track_group(
    viewer,
    endpoint_groups.new_failure_tracks,
    track_name="New Tracks",
    point_name="New Centroids",
    color="lime",
    scale=SCALE_TZYX,
)

add_track_group(
    viewer,
    endpoint_groups.boundary_entry_tracks,
    track_name="Boundary Entry Tracks",
    point_name="Boundary Entry Centroids",
    color="cyan",
    scale=SCALE_TZYX,
    visible=SHOW_BOUNDARY_TRACKS,
)

add_track_group(
    viewer,
    endpoint_groups.boundary_exit_tracks,
    track_name="Boundary Exit Tracks",
    point_name="Boundary Exit Centroids",
    color="orange",
    scale=SCALE_TZYX,
    visible=SHOW_BOUNDARY_TRACKS,
)


napari.run()
